## 02 — MSD Data Preprocessing


In [1]:
import os
import math
import glob
import shutil
import h5py
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, lower, trim, concat_ws, count
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import when, row_number
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.functions import vector_to_array

## THE NEXT TWO CELLS ARE CRUCIAL FOR RUNNING HADOOP AND PYSPARK ON MY DEVICE


In [2]:
import sys

# On Windows, the system-level SPARK_HOME may point to a different Spark version.
# We override it here to make sure PySpark always finds its own bundled JARs.
os.environ["SPARK_HOME"]  = os.path.dirname(pyspark.__file__)

# Spark needs Hadoop's winutils.exe on Windows to handle local file operations
# (directory creation, temp files, etc.). We installed winutils at C:\hadoop\bin\.
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Prepend C:\hadoop\bin to PATH so the JVM can find hadoop.dll via
# System.loadLibrary("hadoop"). HADOOP_HOME alone only locates winutils.exe
# for Hadoop's shell helpers — it does NOT add the directory to
# java.library.path. Without this, NativeIO$Windows.access0 throws
# UnsatisfiedLinkError on Spark write operations.
hadoop_bin = r"C:\hadoop\bin"
if hadoop_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")

# Pin the Python interpreter for both the driver and the executor-side workers
# to the exact Python running this notebook. Without these, PySpark on Windows
# can pick a different Python for workers, causing them to die with
# "Connection reset" the moment Spark tries to deserialize Python rows.
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("SPARK_HOME    :", os.environ["SPARK_HOME"])
print("HADOOP_HOME   :", os.environ["HADOOP_HOME"])
print("PATH[0]       :", os.environ["PATH"].split(os.pathsep)[0])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])

SPARK_HOME    : c:\Users\Mariam\anaconda3\envs\spotify-recommender\Lib\site-packages\pyspark
HADOOP_HOME   : C:\hadoop
PATH[0]       : C:\hadoop\bin
PYSPARK_PYTHON: c:\Users\Mariam\anaconda3\envs\spotify-recommender\python.exe


In [3]:
# local[*]  — uses all CPU cores on this machine (pseudo-distributed mode).
# spark.driver.memory — heap given to the JVM driver process.
# spark.local.dir     — MUST be on the same drive as the output directory (D:).
#   Spark writes temp files here, then renames them to the final output path.
#   Java's File.renameTo() cannot rename across different drives on Windows,
#   so if temp is on C: and output is on D:, every write fails silently.
# spark.hadoop.home.dir — tells the JVM where winutils lives at runtime.
# RawLocalFileSystem  — skips Unix chmod/chown calls that crash on Windows.
# fileoutputcommitter.algorithm.version=2 — avoids the rename-based commit phase.
# marksuccessfuljobs=false — skips writing the _SUCCESS marker file.

SPARK_TEMP_DIR = "D:/tmp/spark"
os.makedirs(SPARK_TEMP_DIR, exist_ok=True)

spark = SparkSession.builder \
    .appName("SpotifyPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.local.dir", SPARK_TEMP_DIR) \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.hadoop.fs.file.impl.disable.cache", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


def spark_write_csv(df, relative_path):
    """
    Write a Spark DataFrame to a single CSV file on Windows.
    Deletes the output directory with Python first so Spark never has to
    rename or overwrite an existing folder across drives.
    """
    if os.path.exists(relative_path):
        shutil.rmtree(relative_path)
    abs_path = os.path.abspath(relative_path).replace("\\", "/")
    df.coalesce(1).write.csv(f"file:///{abs_path}", header=True)
    print(f"Saved → {relative_path}/")

Spark version: 3.5.8


In [4]:
MSD_RAW_DIR   = "../data/raw/million_song/MillionSongSubset"
PROCESSED_DIR = "../data/processed"
SPLIT_DIR     = "../outputs/train_test_split"

# Numeric features to min-max scale. Energy is added dynamically in Step 3
# only if the column is not all-zero.
MSD_NUMERIC_FEATURES = [
    "duration", "tempo", "loudness",
    "artist_familiarity", "artist_hotttnesss", "song_hotttnesss",
]


In [5]:
msd_files = glob.glob(os.path.join(MSD_RAW_DIR, "**", "*.h5"), recursive=True)
print(f"Discovered {len(msd_files):,} MSD .h5 files")
print("Sample paths:")
for p in msd_files[:3]:
    print("  ", p)


Discovered 10,000 MSD .h5 files
Sample paths:
   ../data/raw/million_song/MillionSongSubset\A\A\A\TRAAAAW128F429D538.h5
   ../data/raw/million_song/MillionSongSubset\A\A\A\TRAAABD128F429CF47.h5
   ../data/raw/million_song/MillionSongSubset\A\A\A\TRAAADZ128F9348C2E.h5


In [6]:
# Explicit schema so Spark skips inference (inference would re-scan all 10k files).
MSD_SCHEMA = StructType([
    StructField("track_id",           StringType(), True),
    StructField("track_name",         StringType(), True),
    StructField("artist_name",        StringType(), True),
    StructField("duration",           DoubleType(), True),
    StructField("tempo",              DoubleType(), True),
    StructField("loudness",           DoubleType(), True),
    StructField("artist_familiarity", DoubleType(), True),
    StructField("artist_hotttnesss",  DoubleType(), True),
    StructField("song_hotttnesss",    DoubleType(), True),
])


def _extract_partition(paths_iter):
    """Worker-side: open each .h5 file, pull the row-zero values we want.

    h5py returns bytes for string fields; we decode to UTF-8.
    Numeric fields that come back as NaN are converted to None so Spark stores
    them as proper SQL NULL — NaN would survive dropna and poison MinMaxScaler.
    """
    import h5py
    import math as _math

    def _s(v):
        if v is None:
            return None
        if isinstance(v, bytes):
            v = v.decode("utf-8", errors="replace")
        v = v.strip()
        return v if v else None

    def _f(v):
        if v is None:
            return None
        try:
            f = float(v)
        except (TypeError, ValueError):
            return None
        if _math.isnan(f) or _math.isinf(f):
            return None
        return f

    for path in paths_iter:
        try:
            with h5py.File(path, "r") as f:
                a = f["analysis"]["songs"]
                m = f["metadata"]["songs"]
                yield (
                    _s(a["track_id"][0]),
                    _s(m["title"][0]),
                    _s(m["artist_name"][0]),
                    _f(a["duration"][0]),
                    _f(a["tempo"][0]),
                    _f(a["loudness"][0]),
                    _f(m["artist_familiarity"][0]),
                    _f(m["artist_hotttnesss"][0]),
                    _f(m["song_hotttnesss"][0]),
                )
        except Exception as e:
            # One bad file shouldn't abort the whole job.
            print(f"[skip] {path}: {e}")
            continue


n_partitions = max(8, (os.cpu_count() or 4) * 4)
paths_rdd    = spark.sparkContext.parallelize(msd_files, numSlices=n_partitions)
records_rdd  = paths_rdd.mapPartitions(_extract_partition)

msd_df_raw = spark.createDataFrame(records_rdd, schema=MSD_SCHEMA)
msd_df_raw.cache()
print(f"Extracted {msd_df_raw.count():,} MSD records")

msd_df_raw.printSchema()
msd_df_raw.show(5, truncate=False)


Extracted 10,000 MSD records
root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- artist_familiarity: double (nullable = true)
 |-- artist_hotttnesss: double (nullable = true)
 |-- song_hotttnesss: double (nullable = true)

+------------------+----------------+----------------+---------+-------+--------+------------------+-------------------+------------------+
|track_id          |track_name      |artist_name     |duration |tempo  |loudness|artist_familiarity|artist_hotttnesss  |song_hotttnesss   |
+------------------+----------------+----------------+---------+-------+--------+------------------+-------------------+------------------+
|TRAAAAW128F429D538|I Didn't Mean To|Casual          |218.93179|92.198 |-11.197 |0.5817937658450281|0.4019975433642836 |0.6021199899057548|
|TRAAABD128F429CF

In [7]:
print("step 1: dropping rows missing essential identifiers...")
msd_df = msd_df_raw.dropna(subset=["track_id", "track_name", "artist_name"])

print("step 2: dropping duplicate track_ids...")
msd_df = msd_df.dropDuplicates(["track_id"])

print("step 3: dropping (track_name, artist_name) duplicates...")
msd_df = msd_df.dropDuplicates(["track_name", "artist_name"])

print("step 4: lowercasing + trimming track_name and artist_name...")
msd_df = (
    msd_df
    .withColumn("track_name",  lower(trim(col("track_name"))))
    .withColumn("artist_name", lower(trim(col("artist_name"))))
)

print("step 5: filtering duration to [30s, 15min]...")
msd_df = msd_df.filter((col("duration") >= 30.0) & (col("duration") <= 900.0))

msd_df = msd_df.cache()
print(f"Rows after structural cleaning: {msd_df.count():,}")
msd_df.show(5, truncate=False)


step 1: dropping rows missing essential identifiers...
step 2: dropping duplicate track_ids...
step 3: dropping (track_name, artist_name) duplicates...
step 4: lowercasing + trimming track_name and artist_name...
step 5: filtering duration to [30s, 15min]...
Rows after structural cleaning: 9,851
+------------------+-----------------------+--------------+---------+-------+--------+-------------------+-------------------+------------------+
|track_id          |track_name             |artist_name   |duration |tempo  |loudness|artist_familiarity |artist_hotttnesss  |song_hotttnesss   |
+------------------+-----------------------+--------------+---------+-------+--------+-------------------+-------------------+------------------+
|TRAPTGG128F9326020|34 blues               |charlie patton|173.322  |100.795|-30.53  |0.5743000682773265 |0.3755935825851729 |NULL              |
|TRARHKQ128F92EF0B2|a veces hablo de ti    |tito allen    |322.21995|183.636|-14.305 |0.41448820841052364|0.25113126654

In [8]:
from pyspark.ml.feature import Imputer

# 1. Snapshot null mask so we can restore genuine missingness after scaling.
for c in MSD_NUMERIC_FEATURES:
    msd_df = msd_df.withColumn(f"_null_{c}", col(c).isNull())

# 2. Mean-impute — gives MinMaxScaler a complete vector to fit on.
imputer = Imputer(strategy="mean",
                  inputCols=MSD_NUMERIC_FEATURES,
                  outputCols=MSD_NUMERIC_FEATURES)
msd_df = imputer.fit(msd_df).transform(msd_df)

# 3. Assemble → fit MinMaxScaler → transform → unpack.
assembler    = VectorAssembler(inputCols=MSD_NUMERIC_FEATURES, outputCol="features_vec")
assembled    = assembler.transform(msd_df)
scaler_model = MinMaxScaler(inputCol="features_vec", outputCol="features_scaled").fit(assembled)
scaled       = scaler_model.transform(assembled).withColumn("_arr", vector_to_array("features_scaled"))

for i, c in enumerate(MSD_NUMERIC_FEATURES):
    scaled = scaled.withColumn(c, col("_arr")[i])

# 4. Drop helper columns — keep the imputed scaled values (no nulls).
# Restoring nulls was removed: saving null feature values to CSV causes
# VectorAssembler to crash downstream with handleInvalid = "error".
for c in MSD_NUMERIC_FEATURES:
    scaled = scaled.drop(f"_null_{c}")

msd_df_cleaned = scaled.drop("features_vec", "features_scaled", "_arr")

print(f"Final shape: {msd_df_cleaned.count():,} rows × {len(msd_df_cleaned.columns)} cols")
msd_df_cleaned.printSchema()
msd_df_cleaned.show(5, truncate=False)

Final shape: 9,851 rows × 9 cols
root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- artist_familiarity: double (nullable = true)
 |-- artist_hotttnesss: double (nullable = true)
 |-- song_hotttnesss: double (nullable = true)

+------------------+-----------------------+--------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|track_id          |track_name             |artist_name   |duration           |tempo              |loudness           |artist_familiarity |artist_hotttnesss  |song_hotttnesss    |
+------------------+-----------------------+--------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|TRAPTGG128F9326020|34 blues     

In [9]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
spark_write_csv(msd_df_cleaned, f"{PROCESSED_DIR}/msd_cleaned")


Saved → ../data/processed/msd_cleaned/


In [10]:
spark.stop()
print("Done. Spark session closed.")


Done. Spark session closed.
